In [ ]:
from typing import Annotated, Any

from typing_extensions import TypedDict

from kagraph import Command, END, START, StateGraph, add_messages, invoke_llm
from kagraph.llms import load_llm
from kagraph.messages import AnyMessage, HumanMessage
from kagraph.prebuilt import ToolNode, tools_condition
from kagraph.tracing import trace

In [ ]:
class TeamInput(TypedDict):
    task: str


class TeamState(TypedDict, total=False):
    task: str
    messages: Annotated[list[AnyMessage], add_messages]
    research: str
    chart: str


class AgentInput(TypedDict):
    task: str


class AgentState(TypedDict, total=False):
    messages: Annotated[list[AnyMessage], add_messages]

In [ ]:
def search_product_metrics(query: str) -> str:
    """Return local product metrics for the collaboration example."""

    return (
        "Product retention metrics by plan:\n"
        "- Free: 42%\n"
        "- Pro: 68%\n"
        "- Enterprise: 81%\n"
        "Expansion revenue is strongest in Enterprise."
    )


def make_ascii_bar_chart(title: str, free: int, pro: int, enterprise: int) -> str:
    """Create a compact ASCII bar chart from retention percentages."""

    values = [
        ("Free", int(free)),
        ("Pro", int(pro)),
        ("Enterprise", int(enterprise)),
    ]
    lines = [title]
    for label, value in values:
        lines.append(f"{label:<10} | {'#' * max(1, value // 5)} {value}%")
    return "\n".join(lines)

In [ ]:
def _message_text(message: AnyMessage) -> str:
    text = getattr(message, "text", None)
    if text is not None:
        return str(text)
    return str(getattr(message, "content", ""))


def _last_assistant_text(messages: list[AnyMessage]) -> str:
    for message in reversed(messages):
        if getattr(message.sender, "role", "") == "assistant":
            text = _message_text(message).strip()
            if text:
                return text
    return ""


def _tool_outputs(messages: list[AnyMessage]) -> list[str]:
    outputs: list[str] = []
    for message in messages:
        if getattr(message.sender, "role", "") != "tool":
            continue
        output = getattr(message.content, "output", None)
        if output is not None:
            outputs.append(str(output))
    return outputs

In [ ]:
def build_tool_agent(llm: Any, *, name: str, tools: list[Any], system: str):
    graph = StateGraph(AgentState, input_schema=AgentInput)

    def prepare_task(state: AgentInput):
        return {"messages": [HumanMessage(state["task"])]}

    def agent_node(state: AgentState):
        response = invoke_llm(
            llm,
            messages=state["messages"],
            prompt=(
                "Work on the assigned task. Use your tools when they are useful. "
                "When you have enough information, return a concise final response."
            ),
            system=system,
            tools=tools,
        )
        return {"messages": [response]}

    graph.add_node("prepare_task", prepare_task, input_schema=AgentInput)
    graph.add_node("agent", agent_node)
    graph.add_node("tools", ToolNode(tools))

    graph.add_edge(START, "prepare_task")
    graph.add_edge("prepare_task", "agent")
    graph.add_conditional_edges(
        "agent",
        tools_condition,
        {
            "tools": "tools",
            END: END,
        },
    )
    graph.add_edge("tools", "agent")

    return graph.compile(name=name)

In [ ]:
def build_graph(llm: Any):
    researcher_agent = build_tool_agent(
        llm,
        name="researcher_agent",
        tools=[search_product_metrics],
        system=(
            "You are the research agent in a multi-agent team. You can only do "
            "research. Use search_product_metrics to gather exact plan-retention "
            "data, then summarize the findings for a chart-generator colleague."
        ),
    )
    chart_agent = build_tool_agent(
        llm,
        name="chart_agent",
        tools=[make_ascii_bar_chart],
        system=(
            "You are the chart-generator agent in a multi-agent team. You can only "
            "create charts and interpret them. Use make_ascii_bar_chart with the "
            "numeric retention values, then return the chart and one concise insight."
        ),
    )

    graph = StateGraph(TeamState, input_schema=TeamInput)

    def prepare_task(state: TeamInput):
        return {
            "task": state["task"],
            "messages": [HumanMessage(state["task"])],
        }

    def researcher_node(state: TeamState):
        result = researcher_agent.invoke(
            {
                "task": (
                    "Research this request for the chart generator:\n\n"
                    f"{state['task']}\n\n"
                    "Find product retention metrics for Free, Pro, and Enterprise plans."
                )
            },
            chat_name="KaGraph Research Agent example",
            recursion_limit=8,
        )
        research = _last_assistant_text(result["messages"])
        if not research:
            research = "\n\n".join(_tool_outputs(result["messages"]))
        return {
            "research": research,
            "messages": [("assistant", f"Researcher findings:\n{research}")],
        }

    researcher_node.__kagraph_subgraph__ = researcher_agent

    def chart_node(state: TeamState):
        result = chart_agent.invoke(
            {
                "task": (
                    "Create a simple ASCII retention chart from this research, "
                    "then add one sentence of interpretation:\n\n"
                    f"{state['research']}"
                )
            },
            chat_name="KaGraph Chart Agent example",
            recursion_limit=8,
        )
        chart_parts = [*_tool_outputs(result["messages"]), _last_assistant_text(result["messages"])]
        chart = "\n\n".join(part for part in chart_parts if part.strip())
        return {
            "chart": chart,
            "messages": [("assistant", f"Chart generator output:\n{chart}")],
        }

    chart_node.__kagraph_subgraph__ = chart_agent

    graph.add_node("prepare_task", prepare_task, input_schema=TeamInput)
    graph.add_node("researcher", researcher_node)
    graph.add_node("chart_generator", chart_node)

    graph.add_edge(START, "prepare_task")
    graph.add_edge("prepare_task", "researcher")
    graph.add_edge("researcher", "chart_generator")
    graph.add_edge("chart_generator", END)

    return graph.compile(name="multi_agent_collaboration")

In [ ]:
llm = load_llm("qwen/qwen3-next-80b-a3b-instruct")
app = build_graph(llm)

In [ ]:
app.get_graph().draw_png()

In [ ]:
with trace("MultiAgentCollaboration", include_agent_binary=True):
    result = app.invoke(
        {
            "task": (
                "Compare retention across Free, Pro, and Enterprise plans, "
                "then create a simple chart."
            )
        },
        chat_name="KaGraph Multi-Agent Collaboration example",
        recursion_limit=12,
    )